# Reddit Preprocessing and Topic Modeling
<!-- structured-notebook -->
## Notebook Summary
Purpose: preprocess the merged Reddit corpus and fit the production BERTopic model (Round 11) that powers all downstream Reddit analyses.

Main steps:
- Load the raw JSONL and clean text (URL, emoji, Reddit-syntax removal).
- Filter to English, enforce minimum token length, lemmatize with spaCy.
- Deduplicate identical preprocessed strings; save `original → unique` and `unique → original` index maps.
- Embed with `all-mpnet-base-v2` (L2-normalized), reduce with UMAP, cluster with HDBSCAN (`cluster_selection_method="leaf"`).
- Fit BERTopic with c-TF-IDF; save model, probabilities, and topic assignments under `round_{ROUND}/`.


# Inputs / Outputs
<!-- io-table -->

| Role | File | Upstream / downstream |
|---|---|---|
| Input | `<DATA_ROOT>/Reddit/output/merged_submissions.jsonl` | Produced by `01_data_collection/01_reddit_data_extraction.ipynb` |
| Output | `<DATA_ROOT>/Reddit/output/bertopic/round_{ROUND}/bertopic_no_embed/` | Consumed by `03_topic_refinement/*`, `04_topic_matching/01_cross_platform_matching.ipynb` |
| Output | `<DATA_ROOT>/Reddit/output/bertopic/round_{ROUND}/kept_metadata.parquet` | Consumed by `02_timestamp_attachment.ipynb`, downstream analysis |
| Output | `<DATA_ROOT>/Reddit/output/bertopic/round_{ROUND}/manifest.json` | Single source of truth for hyperparameters |
| Output | `<DATA_ROOT>/Reddit/output/preprocessed-docs.pkl` | Consumed by `03_topic_refinement/01_subtopic_analysis.ipynb`, BTM |


# Live Hyperparameter Reference
<!-- live-hyperparams -->

Production hyperparameters are persisted to `round_11/manifest.json` by this notebook. Load the manifest below to see the current values — this is the single source of truth. If you are training a new round, update the configuration cell further down and the manifest will be regenerated automatically.


In [ ]:
import json
import pandas as pd
from src.shared_reddit_telegram.config import REDDIT_MODELS

manifest_path = REDDIT_MODELS / "round_11" / "manifest.json"
manifest = json.loads(manifest_path.read_text())
pd.json_normalize(manifest).T.rename(columns={0: "value"})


# Why These Choices
<!-- structured-notebook -->
## Summary

**Embedding: `all-mpnet-base-v2` over a Reddit-specific transformer.**
A Reddit-specific sentence transformer was tried first. It required more compute than was available locally, so the pipeline uses the general-purpose `all-mpnet-base-v2` with L2 normalization. Using the same embedder across Reddit, PubMed, and Telegram also makes Bidirectional Topic Matching (BTM) cleaner — comparable embedding geometries remove a confound.

**Caveat on reload:** when loading a saved BERTopic model that was trained with L2-normalized embeddings, the embeddings must be **renormalized at transform time** before `.transform()` is called, or topic assignments will be off.

**Clustering: HDBSCAN with `cluster_selection_method="leaf"`.**
Initial v1 runs used partial-fit BERTopic with MiniBatchKMeans because the 9-subreddit corpus (~1.4M posts) did not fit in memory. After the corpus was narrowed to the three most relevant subreddits (`Biohackers`, `longevity`, `Aging` → 79,152 posts → 72,806 unique), HDBSCAN became viable. Round 11 is the only round using `leaf` (all earlier rounds used `eom`); leaf produces finer-grained clusters and won on interpretability.

**UMAP `min_dist=0.0`.**
This was the fix for a late-stage hiccup where topics were failing to separate cleanly. Setting `min_dist=0.0` makes the embedding space more clusterable.

**Vectorizer: `ngram_range=(1,2)`, `min_df=3`, `max_df=0.5`, `max_features=40000`.**
Earlier rounds used higher `max_df` that let generic longevity-domain terms dominate. Round 11 lowered it to 0.5.


# Deduplication Guardrails
<!-- structured-notebook -->
## Summary
Deduplication was added after a bug caused topic-doc mappings to be misaligned. The fix saves both `map_orig_to_unique.npy` and `unique_to_orig.pkl` so every original doc can still be routed to its topic.

**Always-apply rules when working with `kept_indices`:**

```python
docs = pd.read_json(input_path, lines=True).reset_index(drop=True)
# Use .iloc, never label-based indexing:
subset = docs.iloc[kept_indices]
assert kept_idx.max() < len(df_raw), "kept_indices out of range for current raw DF"
```

Forgetting the `reset_index(drop=True)` silently misaligns rows because Pandas preserves the source index across the JSON read.

### Saved artifacts after preprocessing

| File | Contents |
|---|---|
| `kept_indices.npy` | Docs that survived preprocessing; indices into the raw JSONL |
| `kept_ts_seconds.npy` | UTC timestamps for kept indices |
| `map_orig_to_unique.npy` | kept docs → unique doc (post-deduplication) |
| `preprocessed-docs.pkl` | Kept docs, pre-dedup |
| `unique_docs.pkl` | Kept, post-dedup docs |
| `unique_to_orig.pkl` | unique doc → kept docs (many-to-one) |


# Round Progression
<!-- structured-notebook -->
## Summary
This notebook is `ROUND`-parameterized; changing the `ROUND` constant in the configuration cell and re-running produces a new `round_{ROUND}/` directory. Round 11 is production; earlier rounds are retained under the per-round artifact folders for comparison.

| Round | What changed | Outcome |
|---|---|---|
| 4 | 40k max features, saving artifacts for downstream analysis; no soft probs | Too few topics, some dominant + vague |
| 5 | Parameter tuning + bugfixes | Intermediate |
| 6–8 | Further tuning to reduce outliers, push toward finer topics | Improving |
| 9 | More tuning toward finer topics | Good noise control |
| 10 | 78 topics, 30% noise, `eom` | Production candidate |
| **11** | `leaf` selection, soft probabilities, UMAP `min_dist=0.0` | **Production** |

Run the `utilities/01_parameter_tuning.ipynb` harness to sweep parameter combinations automatically.


## 1. Setup & Imports

We import shared utilities from the project's `src/shared_reddit_telegram/` module and set random seeds for full reproducibility across runs.

In [ ]:
import sys
import os
import json
import random
import pickle
import logging
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm
from langdetect import DetectorFactory

# Walk up from the notebook's directory to the repo root (the first ancestor that contains `src/`) and add it to sys.path
_repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))
from src.shared_reddit_telegram.text_cleaning import clean_text, is_english, lemmatize_texts, process_docs, dedupe_strings, preprocess_pipeline
from src.shared_reddit_telegram.config import PROJECT_ROOT, REDDIT_DATA, REDDIT_SMALL_DATA

# Reproducibility
np.random.seed(0)
random.seed(0)
DetectorFactory.seed = 0

# Logging
logging.basicConfig(level=logging.INFO, format='[%(asctime)s] %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print(f"Project root: {PROJECT_ROOT}")

## 2. Configuration

All tunable parameters are collected here for easy experimentation.

| Parameter | Purpose |
|---|---|
| `ROUND` | Tuning iteration number; outputs are saved under `round_{ROUND}/` |
| `RAW_PATH` | Path to the merged JSONL of raw Reddit submissions |
| `UMAP_*` | Dimensionality reduction hyperparameters |
| `HDBSCAN_*` | Clustering hyperparameters |

In [ ]:
ROUND = 11  # Current tuning round
ROUND_PATH = f"../scripts/topic_modelling_v2/round_{ROUND}"
RAW_PATH = str(REDDIT_SMALL_DATA / "merged_submissions.jsonl")

# UMAP parameters
UMAP_N_NEIGHBORS = 12
UMAP_N_COMPONENTS = 12
UMAP_MIN_DIST = 0.0
UMAP_METRIC = "cosine"

# HDBSCAN parameters
HDBSCAN_MIN_CLUSTER_SIZE = 146
HDBSCAN_MIN_SAMPLES = 10
HDBSCAN_METRIC = "euclidean"
HDBSCAN_SELECTION_METHOD = "leaf"

# Vectorizer parameters
NGRAM_RANGE = (1, 2)
MIN_DF = 3
MAX_DF = 0.5
MAX_FEATURES = 40_000

# Embedding model
EMBEDDING_MODEL = "all-mpnet-base-v2"

os.makedirs(ROUND_PATH, exist_ok=True)
print(f"Outputs will be saved to: {ROUND_PATH}")
print(f"Raw data: {RAW_PATH}")

## 3. Load Raw Data

Read the merged JSONL file containing Reddit submissions. Each row has `title`, `selftext`, `created_utc` (Unix epoch seconds), plus metadata like `subreddit`, `score`, and `num_comments`.

We convert `created_utc` into both a UTC datetime and an integer seconds column for downstream temporal analysis.

In [ ]:
df_raw = pd.read_json(RAW_PATH, lines=True).reset_index(drop=True)
print(f"Loaded {len(df_raw):,} raw submissions")
print(f"Columns: {list(df_raw.columns)}")

# Extract timestamps
ts_col = "created_utc"
if ts_col not in df_raw.columns:
    df_raw[ts_col] = np.nan

ts = pd.to_datetime(df_raw[ts_col], unit="s", utc=True, errors="coerce")
ts_seconds = ts.view("int64") // 10**9
ts_seconds = ts_seconds.where(ts.notna(), other=-1).astype(np.int64)
df_raw["_ts_seconds"] = ts_seconds

print(f"Date range: {ts.min()} to {ts.max()}")
df_raw.head(3)

## 4. Preprocess Documents

The preprocessing pipeline applies multiple stages:

1. **Concatenation**: `title + " " + selftext` to form each document.
2. **URL removal**: Strips `http://`, `https://`, and `www.` links.
3. **Emoji removal**: Replaces all Unicode emojis with empty string.
4. **Reddit syntax cleanup**: Removes `r/subreddit`, `u/username`, and blockquotes (`> ...`).
5. **Character filtering (v2)**: Keeps `\w`, hyphens, `+`, `/`, `#`, apostrophes. This is critical for preserving supplement names like `B12`, `NAD+`, `5-HTP`, `BPC-157`.
6. **Language detection**: Filters out non-English posts using `langdetect`.
7. **Minimum length**: Discards posts with 3 or fewer tokens after cleaning.
8. **Lemmatization**: spaCy `en_core_web_sm` with stopword and punctuation removal.
9. **Deduplication**: Identical preprocessed strings are collapsed; a mapping is kept to expand back.

In [ ]:
# Concatenate title + selftext into document strings
raw_texts = process_docs(df_raw)
print(f"Combined {len(raw_texts):,} documents (title + selftext)")

# Full preprocessing: clean -> English filter -> min-length -> lemmatize -> deduplicate
preprocessed_docs, kept_indices, unique_docs, map_o2u = preprocess_pipeline(
    raw_texts,
    strip_numbers=False,  # v2: preserve supplement names
    min_tokens=3,
    spacy_model="en_core_web_sm",
)

print(f"\nPreprocessing summary:")
print(f"  Raw documents:       {len(raw_texts):,}")
print(f"  After cleaning:      {len(preprocessed_docs):,} ({len(preprocessed_docs)/len(raw_texts):.1%})")
print(f"  Unique (deduplicated): {len(unique_docs):,}")
print(f"  Duplicate ratio:     {1 - len(unique_docs)/len(preprocessed_docs):.3f}")

In [ ]:
# Save preprocessing artifacts
kept_ts = df_raw["_ts_seconds"].to_numpy(np.int64)[kept_indices]

np.save(f"{ROUND_PATH}/kept_indices.npy", np.asarray(kept_indices, dtype=np.int64))
np.save(f"{ROUND_PATH}/kept_ts_seconds.npy", kept_ts)
np.save(f"{ROUND_PATH}/map_orig_to_unique.npy", map_o2u)

with open(f"{ROUND_PATH}/unique_docs.pkl", "wb") as f:
    pickle.dump(unique_docs, f)
with open(f"{ROUND_PATH}/preprocessed-docs.pkl", "wb") as f:
    pickle.dump(preprocessed_docs, f)

print(f"Saved preprocessing outputs to {ROUND_PATH}/")

## 5. Embedding Generation

We embed the **unique** documents (not duplicates) using `all-mpnet-base-v2`, a 768-dimensional sentence transformer that performs well on semantic similarity tasks.

**L2 normalization** is applied during encoding (`normalize_embeddings=True`), which:
- Puts all embeddings on the unit hypersphere
- Makes cosine distance equivalent to Euclidean distance (up to a constant), simplifying downstream UMAP/HDBSCAN

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBEDDING_MODEL)
embeddings = embedder.encode(
    unique_docs,
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True,
).astype("float32")

print(f"Embedding shape: {embeddings.shape}")
print(f"Mean L2 norm: {np.linalg.norm(embeddings, axis=1).mean():.4f} (should be ~1.0)")

np.save(f"{ROUND_PATH}/embeddings_fp32_l2.npy", embeddings)

## 6. UMAP Dimensionality Reduction

UMAP reduces the 768-dimensional embeddings to a lower-dimensional space for clustering.

Key parameter choices:
- **`metric="cosine"`**: Natural similarity metric for normalized sentence embeddings.
- **`n_neighbors=12`**: Controls the locality of the manifold approximation. Smaller values preserve more local structure; larger values capture more global patterns. 12 balances detail and stability for ~60K docs.
- **`n_components=12`**: The target dimensionality. Higher than the typical 5 used for visualization, because we want to preserve enough structure for HDBSCAN to find meaningful density clusters.
- **`min_dist=0.0`**: Packs points as tightly as possible, which helps HDBSCAN identify dense clusters.

In [ ]:
import umap

umap_model = umap.UMAP(
    n_neighbors=UMAP_N_NEIGHBORS,
    n_components=UMAP_N_COMPONENTS,
    min_dist=UMAP_MIN_DIST,
    metric=UMAP_METRIC,
    random_state=42,
    low_memory=True,
    verbose=True,
)

print(f"UMAP configured: {UMAP_N_NEIGHBORS} neighbors, {UMAP_N_COMPONENTS}D, {UMAP_METRIC} metric")

## 7. HDBSCAN Clustering

HDBSCAN is a density-based clustering algorithm that automatically determines the number of clusters. It runs in the **UMAP-reduced space** (hence `metric="euclidean"` rather than cosine).

Key parameter choices:
- **`min_cluster_size=146`**: The minimum number of documents to form a cluster. Too small creates noisy micro-topics; too large merges distinct themes. For ~60K unique docs, 146 (~0.2%) is a good balance.
- **`min_samples=10`**: Controls how conservative the clustering is. Higher values make it harder for border points to join clusters.
- **`cluster_selection_method="leaf"`**: Selects the finest-grained clusters from the hierarchy. The alternative `"eom"` (excess of mass) tends to produce fewer, larger clusters.

In [ ]:
import hdbscan

hdbscan_model = hdbscan.HDBSCAN(
    min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
    min_samples=HDBSCAN_MIN_SAMPLES,
    metric=HDBSCAN_METRIC,
    cluster_selection_method=HDBSCAN_SELECTION_METHOD,
    prediction_data=True,
)

print(f"HDBSCAN configured: min_cluster_size={HDBSCAN_MIN_CLUSTER_SIZE}, method={HDBSCAN_SELECTION_METHOD}")

## 8. Fit BERTopic

BERTopic ties the pipeline together: it runs UMAP, then HDBSCAN, then extracts topic representations using class-based TF-IDF (c-TF-IDF) over the discovered clusters.

The `CountVectorizer` controls the vocabulary for c-TF-IDF:
- **`ngram_range=(1, 2)`**: Captures both unigrams and bigrams (e.g., "vitamin d", "sleep quality")
- **`min_df=3`**: Terms must appear in at least 3 documents
- **`max_df=0.5`**: Drops terms appearing in more than 50% of documents (too generic)
- **`max_features=40,000`**: Caps vocabulary to prevent memory issues

In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
    stop_words="english",
    ngram_range=NGRAM_RANGE,
    min_df=MIN_DF,
    max_df=MAX_DF,
    max_features=MAX_FEATURES,
    dtype=np.float32,
    strip_accents="unicode",
)

topic_model = BERTopic(
    embedding_model=embedder,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    nr_topics=None,
    top_n_words=10,
    calculate_probabilities=True,
    verbose=True,
)

topics, probs = topic_model.fit_transform(unique_docs, embeddings=embeddings)

print(f"\nFitting complete.")
print(f"Number of topics discovered: {len(set(topics)) - (1 if -1 in topics else 0)}")
print(f"Noise ratio: {np.mean(np.array(topics) == -1):.3f}")

## 9. Save Outputs

We save several artifacts:

| File | Description |
|---|---|
| `topics.csv` | Topic info table (ID, count, top words) |
| `train_topics_unique.npy` | Topic assignment per unique doc |
| `train_topics_original.npy` | Topic assignment expanded back to all kept docs (including duplicates) |
| `bertopic_no_embed/` | BERTopic model (without embedder, to save space) |
| `embedder/` | SentenceTransformer saved separately |
| `kept_metadata.parquet` | Full metadata table with topic assignments, timestamps, subreddit info |

In [ ]:
# Save topic assignments for unique and original (with duplicates) documents
topics_unique = np.asarray(topics, dtype=np.int32)
np.save(f"{ROUND_PATH}/train_topics_unique.npy", topics_unique)

# Expand topic assignments back to original (pre-dedup) documents
map_o2u_arr = np.asarray(map_o2u, dtype=np.int64)
topics_original = topics_unique[map_o2u_arr]
np.save(f"{ROUND_PATH}/train_topics_original.npy", topics_original)

# Save topic info CSV
topic_info = topic_model.get_topic_info()
topic_info.to_csv(f"{ROUND_PATH}/topics.csv", index=False)

# Save BERTopic model (without embedding model to save disk space)
topic_model.save(f"{ROUND_PATH}/bertopic_no_embed", save_ctfidf=True, save_embedding_model=False)

# Save the SentenceTransformer separately
embedder.save_pretrained(f"{ROUND_PATH}/embedder", safe_serialization=True)

print(f"Model and topic assignments saved to {ROUND_PATH}/")

In [ ]:
# Build and save metadata parquet with topic assignments + raw metadata
kept_idx = np.load(f"{ROUND_PATH}/kept_indices.npy")
ts_K = np.load(f"{ROUND_PATH}/kept_ts_seconds.npy")

need_cols = [
    "subreddit", "score", "num_comments", "author", "link_flair_text",
    "selftext", "title", "permalink", "url", "is_self", "id",
]
for c in need_cols:
    if c not in df_raw.columns:
        df_raw[c] = np.nan

meta = df_raw.iloc[kept_idx][need_cols].reset_index(drop=True)
meta.insert(0, "kept_index", np.arange(len(kept_idx), dtype=np.int32))
meta["orig_index"] = kept_idx.astype(np.int32)
meta["unique_index"] = map_o2u_arr.astype(np.int32)
meta["topic_hard"] = topics_original.astype(np.int32)
meta["ts_seconds"] = ts_K.astype(np.int64)
meta["ts_utc"] = pd.to_datetime(meta["ts_seconds"], unit="s", utc=True, errors="coerce")

# Duplicate group size
dup_sizes = np.bincount(map_o2u_arr, minlength=len(unique_docs)).astype(np.int32)
meta["dup_group_size"] = dup_sizes[meta["unique_index"]].astype(np.int32)

# Compact dtypes
meta["subreddit"] = meta["subreddit"].astype("category")
for c in ("author", "link_flair_text"):
    meta[c] = meta[c].astype("category")

meta.to_parquet(f"{ROUND_PATH}/kept_metadata.parquet", index=False)
print(f"Metadata parquet saved: {len(meta):,} rows, {meta.shape[1]} columns")

In [ ]:
# Save a manifest with all hyperparameters for reproducibility
manifest = {
    "n_docs_original": int(len(df_raw)),
    "n_docs_kept": int(len(preprocessed_docs)),
    "n_docs_unique": int(len(unique_docs)),
    "embedding_model": f"{EMBEDDING_MODEL} (L2-normalized)",
    "umap": {
        "n_neighbors": UMAP_N_NEIGHBORS,
        "n_components": UMAP_N_COMPONENTS,
        "min_dist": UMAP_MIN_DIST,
        "metric": UMAP_METRIC,
        "random_state": 42,
    },
    "hdbscan": {
        "min_cluster_size": HDBSCAN_MIN_CLUSTER_SIZE,
        "min_samples": HDBSCAN_MIN_SAMPLES,
        "metric": HDBSCAN_METRIC,
        "cluster_selection_method": HDBSCAN_SELECTION_METHOD,
    },
    "vectorizer": {
        "ngram_range": list(NGRAM_RANGE),
        "min_df": MIN_DF,
        "max_df": MAX_DF,
        "max_features": MAX_FEATURES,
    },
}
with open(f"{ROUND_PATH}/manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print(json.dumps(manifest, indent=2))

## 10. Quick Inspection

A first look at the discovered topics. Each row shows the topic ID, document count, and representative keywords extracted via c-TF-IDF.

In [ ]:
topic_info = topic_model.get_topic_info()
print(f"Total topics: {len(topic_info)} (including outlier topic -1)")
print(f"Outlier documents: {topic_info.loc[topic_info['Topic'] == -1, 'Count'].values[0]:,}")
print()
topic_info.head(20)

In [ ]:
# Diagnostic stats
labels = topic_model.hdbscan_model.labels_
n_clusters = len(set(labels[labels >= 0]))
noise_ratio = np.mean(labels == -1)

lens = np.array([len(s.split()) for s in unique_docs])

print(f"HDBSCAN clusters: {n_clusters}")
print(f"Noise ratio: {noise_ratio:.3f}")
print(f"BERTopic topics (post-reduction): {(topic_info['Topic'] >= 0).sum()}")
print(f"Document length -- median: {int(np.median(lens))}, p10: {int(np.percentile(lens, 10))}, p90: {int(np.percentile(lens, 90))}")

---
<!-- nav-footer -->

← [01 reddit data extraction](../../../../SocialMedia/Reddit/notebooks/01_data_collection/01_reddit_data_extraction.ipynb) | [Project Overview](../../../../PROJECT_OVERVIEW.ipynb) | [02 reddit topic analysis](../../../../SocialMedia/Reddit/notebooks/02_topic_analysis/02_reddit_topic_analysis.ipynb) →
